In [11]:
from dotenv import load_dotenv
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.agents import AssistantAgent
from autogen_core import CancellationToken

In [2]:
load_dotenv(override=True)

True

In [ ]:
# Step 1: Define the llm model client
model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

In [7]:
# Step 2: Create the message
message = TextMessage(content="I would like to go to London", source="user")
message

TextMessage(id='ea613b2a-5644-4887-89ac-d96c565212c7', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 3, 31, 18, 22, 54, 695846, tzinfo=datetime.timezone.utc), content='I would like to go to London', type='TextMessage')

In [10]:
# Step 3: Create the agent
agent = AssistantAgent(
    name="airline_agent",
    model_client=model_client,
    system_message="You are a helpful assitant for a airline company. You provide short and humorous answers",
    model_client_stream=True
)

In [20]:
# Step 4: Call the llm
resposne = await agent.on_messages(messages=[message], cancellation_token=CancellationToken())
resposne.chat_message.content

'Fantastic! Just remember: if someone offers you a "cuppa," it’s not a dance move, it’s tea! Ready to book?'

In [21]:
# Create a datase with ticket prices which will be used by the tool
import os
import sqlite3


if os.path.exists("tickets.db"):
    os.remove("tickets.db")

conn = sqlite3.connect("tickets.db")
cursor = conn.cursor()
cursor.execute("CREATE TABLE cities (city_name TEXT PRIMARY KEY, round_trip_price REAL)")
conn.commit()
conn.close()

In [ ]:
# Populate the database with some sample ticket prices
def save_city_price(city_name: str, round_trip_price: float) -> None:
    conn = sqlite3.connect("tickets.db")
    cursor = conn.cursor()
    cursor.execute("REPLACE INTO cities (city_name, round_trip_price) VALUES (?, ?)", (city_name.lower(), round_trip_price))
    conn.commit()
    conn.close()

save_city_price("London", 299)
save_city_price("Paris", 399)
save_city_price("Rome", 499)
save_city_price("Madrid", 550)
save_city_price("Barcelona", 580)
save_city_price("Berlin", 525)

In [25]:
# Get the price of the city
def get_city_price(city_name: str) -> float | None:
    conn = sqlite3.connect("tickets.db")
    cursor = conn.cursor()
    cursor.execute("SELECT round_trip_price FROM cities WHERE city_name = ?", (city_name.lower(),))
    result = cursor.fetchone()
    conn.close()

    return result[0] if result else None

In [26]:
get_city_price("Paris")

399.0

In [28]:
smart_agent = AssistantAgent(
    name="smart_airline_agent",
    model_client=model_client,
    tools=[get_city_price],
    system_message="You are a helpful assistant for an airline. You give short, humorous answers, including the price of a roundtrip ticket.",
    model_client_stream=True,
    reflect_on_tool_use=True
)

In [33]:
response = await smart_agent.on_messages(messages=[message], cancellation_token=CancellationToken())

In [36]:
response.chat_message.content

'Absolutely! London awaits with its fog and royal charm! A roundtrip ticket is $299. Just remember to practice your British accent before you land—"Cheerio!" 🏰✈️'

In [38]:
for inner_message in response.inner_messages:
    print(inner_message.content)

[FunctionCall(id='call_QVRJ72PlI0GIRtOx1POz7xFZ', arguments='{"city_name":"London"}', name='get_city_price')]
[FunctionExecutionResult(content='299.0', name='get_city_price', call_id='call_QVRJ72PlI0GIRtOx1POz7xFZ', is_error=False)]
